# AeroPulse — Auto Loader Recovery Test

## Purpose

Validate Auto Loader checkpoint-based recovery after a
controlled streaming failure.

## Test Objective

Prove that:

1. New files are discovered incrementally.
2. A failed run does not require manual file tracking.
3. Restarting with the same checkpoint resumes correctly.
4. Previously committed records are not duplicated.

## Important

This notebook is a controlled failure experiment.
It must not delete or modify the production-style
maintenance checkpoint.

In [0]:
TEST_SOURCE_PATH = (
    "/Volumes/workspace/aeropulse_dev/"
    "raw_landing/maintenance_app/"
    "maintenance_recovery_test"
)

TEST_CHECKPOINT_PATH = (
    "/Volumes/workspace/aeropulse_dev/"
    "raw_landing/_checkpoints/"
    "maintenance_app/"
    "maintenance_recovery_test"
)

TEST_BRONZE_TABLE = (
    "workspace.aeropulse_dev."
    "bronze_maintenance_recovery_test"
)

print(TEST_SOURCE_PATH)
print(TEST_CHECKPOINT_PATH)
print(TEST_BRONZE_TABLE)

In [0]:
%run /Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/source_generators/maintenance_generator.py

In [0]:
recovery_test_df.write \
    .mode("overwrite") \
    .json(TEST_SOURCE_PATH)

In [0]:
test_source_df = (
    spark.read
    .json(TEST_SOURCE_PATH)
)

test_source_df.printSchema()

In [0]:
(
    test_source_df
    .limit(0)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TEST_BRONZE_TABLE)
)

In [0]:
print(
    spark.table(TEST_BRONZE_TABLE).count()
)

In [0]:
recovery_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option(
        "cloudFiles.format",
        "json"
    )
    .option(
        "cloudFiles.schemaLocation",
        TEST_CHECKPOINT_PATH
    )
    .load(TEST_SOURCE_PATH)
)

In [0]:
recovery_bronze_df = (
    recovery_stream_df
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_system",
        F.lit("maintenance_app")
    )
    .withColumn(
        "_source_entity",
        F.lit("maintenance_recovery_test")
    )
    .withColumn(
        "_source_file_path",
        F.col("_metadata.file_path")
    )
)

In [0]:
recovery_query = (
    recovery_bronze_df
    .writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        TEST_CHECKPOINT_PATH
    )
    .option(
        "mergeSchema",
        "true"
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        TEST_BRONZE_TABLE
    )
)

recovery_query.awaitTermination()

In [0]:
print(
    spark.table(TEST_BRONZE_TABLE).count()
)

In [0]:
from datetime import datetime, timezone

recovery_timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

recovery_delivery_path = (
    f"{TEST_SOURCE_PATH}/"
    f"delivery_{recovery_timestamp}"
)

In [0]:
(
    recovery_test_df_2
    .write
    .mode("error")
    .json(recovery_delivery_path)
)

In [0]:
BAD_BRONZE_TABLE = (
    "workspace.aeropulse_dev."
    "bronze_maintenance_recovery_bad"
)

In [0]:
(
    test_source_df
    .limit(0)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BAD_BRONZE_TABLE)
)

In [0]:
bad_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option(
        "cloudFiles.format",
        "json"
    )
    .option(
        "cloudFiles.schemaLocation",
        TEST_CHECKPOINT_PATH
    )
    .load(TEST_SOURCE_PATH)
    .withColumn(
        "THIS_COLUMN_DOES_NOT_EXIST",
        F.col("does_not_exist")
    )
)

In [0]:
# Fixed: recreate stream without the bad column reference
fixed_bad_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option(
        "cloudFiles.format",
        "json"
    )
    .option(
        "cloudFiles.schemaLocation",
        TEST_CHECKPOINT_PATH
    )
    .load(TEST_SOURCE_PATH)
)

bad_query = (
    fixed_bad_stream_df
    .writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        TEST_CHECKPOINT_PATH
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        BAD_BRONZE_TABLE
    )
)

In [0]:
fixed_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option(
        "cloudFiles.format",
        "json"
    )
    .option(
        "cloudFiles.schemaLocation",
        TEST_CHECKPOINT_PATH
    )
    .load(TEST_SOURCE_PATH)
)

In [0]:
fixed_bronze_df = (
    fixed_stream_df
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_system",
        F.lit("maintenance_app")
    )
    .withColumn(
        "_source_entity",
        F.lit("maintenance_recovery_test")
    )
    .withColumn(
        "_source_file_path",
        F.col("_metadata.file_path")
    )
)

In [0]:
fixed_query = (
    fixed_bronze_df
    .writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        TEST_CHECKPOINT_PATH
    )
    .option("mergeSchema", "true")
    .trigger(
        availableNow=True
    )
    .toTable(
        BAD_BRONZE_TABLE
    )
)

fixed_query.awaitTermination()

In [0]:
print(
    spark.table(BAD_BRONZE_TABLE).count()
)

In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM cloud_files_state(
            '{TEST_CHECKPOINT_PATH}'
        )
    """)
)